### Setup

In [1]:
import os
import sys

sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))

from common.utils import DataPreprocessor, FeatureEngineer, set_seed
from common.exp_data_utils import ExperimentDataPreprocessor
from common.eval import Evaluator

MOVIELENS_DATA_DIR = "../datasets/hetrec2011-movielens-2k-v2/user_ratedmovies.dat"
RANDOM_SEED = 42

# Initialize data processors
set_seed(RANDOM_SEED)
data_preprocessor = DataPreprocessor()
feature_engineer = FeatureEngineer()
experiment_data_preprocessor = ExperimentDataPreprocessor()
evaluator = Evaluator()


/home/adam/R11_Bai/DPRecSys/.venv/lib/python3.11/site-packages/transformers/utils/generic.py:441: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(
Seed set to 42
Seed set to 42


random seed set to 42
numpy seed set to 42
torch seed set to 42
lightning seed set to 42
torch set to use deterministic algorithms


### Load and Process DataFrame

In [2]:
interaction_df = data_preprocessor.load_and_process_df(
    file_dir=MOVIELENS_DATA_DIR,
    year_range=(2006, 2008),
)
interaction_df.head()

Data count: 855598
Data count after filtering by year (2006, 2008): 480608
Num of distinct users: 2103
Num of distinct items: 9519
done!
------------------------------
Filtering by min user/item interactions (10/0):
Data count before: 480608
Data count after: 480448
done!
------------------------------
==== Final Data Info: ====
Data Year Range: (2006, 2008)
Rating Threshold: 4.0
Num of interactions: 480448
Num of distinct users: 2064
Num of distinct items: 9519


,userID,movieID,rating,date_day,date_month,date_year,date_hour,date_minute,date_second,timestamp,label
0,75,3,1.0,29,10,2006,23,17,16,2006-10-29 23:17:16,0
1,75,32,4.5,29,10,2006,23,23,44,2006-10-29 23:23:44,1
2,75,110,4.0,29,10,2006,23,30,8,2006-10-29 23:30:08,1
3,75,160,2.0,29,10,2006,23,16,52,2006-10-29 23:16:52,0
4,75,163,4.0,29,10,2006,23,29,30,2006-10-29 23:29:30,1


### Join Side Information

In [3]:
interaction_info_df = data_preprocessor.join_item_features(
    df=interaction_df, actor_k=5, threshold=5
)
interaction_info_df.head()

extracting item features...
merging features...
interaction data count before merging: 480448
interaction data count after merging: 478404
done!


,userID,movieID,rating,date_day,date_month,date_year,date_hour,date_minute,date_second,timestamp,label,actorID,country,directorID,directorName,genre
0,75,3,1.0,29,10,2006,23,17,16,2006-10-29 23:17:16,0,"[jack_lemmon, walter_matthau, annmargret, burg...",USA,donald_petrie,Donald Petrie,"[Comedy, Romance, [PAD], [PAD], [PAD], [PAD], ..."
1,75,32,4.5,29,10,2006,23,23,44,2006-10-29 23:23:44,1,"[[RARE], [RARE], [RARE], [RARE], [RARE]]",USA,[RARE],Siddharth Randeria,"[Sci-Fi, Thriller, [PAD], [PAD], [PAD], [PAD],..."
2,75,110,4.0,29,10,2006,23,30,8,2006-10-29 23:30:08,1,"[mel_gibson, sophie_marceau, patrick_mcgoohan,...",USA,[RARE],Mel Gibson,"[Action, Drama, War, [PAD], [PAD], [PAD], [PAD..."
3,75,160,2.0,29,10,2006,23,16,52,2006-10-29 23:16:52,0,"[[RARE], laura_linney, ernie_hudson_jr, tim_cu...",USA,frank_marshall,Frank Marshall,"[Action, Adventure, Mystery, Sci-Fi, [PAD], [P..."
4,75,163,4.0,29,10,2006,23,29,30,2006-10-29 23:29:30,1,"[antonio_banderas, salma_hayek, 1142520-joaqui...",USA,robert_rodriguez,Robert Rodriguez,"[Action, Romance, Thriller, [PAD], [PAD], [PAD..."


### Prepare Train/Valid/Test Set

In [4]:
# TODO: determine which method to use for splitting
# 1. Split by year
# 2. Stratified split by user, timestamp

train_df, valid_df, test_df = experiment_data_preprocessor.stratified_time_split(
    interaction_info_df,
    time_col="timestamp",
    train_ratio=0.75,
    val_ratio=0.1,
    test_ratio=0.15,
)

TRAIN_NUM_USERS = len(train_df["userID"].unique())
TRAIN_NUM_ITEMS = len(train_df["movieID"].unique())


Splitting data into train/valid/test by time period with ratio=(0.75 : 0.1 : 0.15):
train: 358027 (74.84%)
valid: 46916 (9.81%)
test: 73461 (15.36%)
------------------------------ 

Check target label distribution after splitting (%):
train label
0    0.555944
1    0.444056
Name: proportion, dtype: float64
valid label
0    0.610772
1    0.389228
Name: proportion, dtype: float64
test label
0    0.585277
1    0.414723
Name: proportion, dtype: float64


### Re-index User/Item ID & Encode Categorical Features

In [5]:
print("Train: fit_transform")
encoded_train_df = feature_engineer.fit_transform(train_df)
print("---"*10)
print("Valid: transform")
encoded_valid_df = feature_engineer.transform(valid_df)
print("---"*10)
print("Test: transform")
encoded_test_df = feature_engineer.transform(test_df)
print("---"*10)

Train: fit_transform
Re-index mapping dumped into ...
user: ../datasets/userid_mapping.csv
item: ../datasets/itemid_mapping.csv
Fitted: user/item mapping
Fitted: vocab2idx for actorID
Fitted: vocab2idx for country
Fitted: vocab2idx for directorID
Fitted: vocab2idx for genre
Transformed: Re-index user/item mapping
Transformed: Encoded idx for actorID
Transformed: Encoded idx for country
Transformed: Encoded idx for directorID
Transformed: Encoded idx for genre
------------------------------
Valid: transform
Transformed: Re-index user/item mapping
Transformed: Encoded idx for actorID
Transformed: Encoded idx for country
Transformed: Encoded idx for directorID
Transformed: Encoded idx for genre
------------------------------
Test: transform
Transformed: Re-index user/item mapping
Transformed: Encoded idx for actorID
Transformed: Encoded idx for country
Transformed: Encoded idx for directorID
Transformed: Encoded idx for genre
------------------------------


In [6]:
# # NOTE: can check the encoding vocab idx content from the feature engineer
# oov_idx = feature_engineer.vocab2idx["movieID"]["[OOV]"]
# len(test_df[test_df["movieID"] == oov_idx])

### Prepare Additional Data for Train/Inference

#### Build bi-partite graph for training

In [7]:
# NOTE: At training, we use interaction graph from train_df for train and validation
train_graph = experiment_data_preprocessor.create_interaction_graph(encoded_train_df)

# NOTE: At inference, we can use graph of (train_df + valid_df)
# train_valid_graph = utils.create_interaction_graph(pd.concat([train_df, valid_df], axis=0))

Creating interaction graph...
Drop negative samples
  Num of all interactions: 358027
  Num of positive interactions: 158984 

Building edges...
Building labels...
Interaction Graph: Data(edge_index=[2, 317967], edge_label=[158984])
Edge Index: tensor([[    0,     0,     0,  ..., 10758, 10762, 10763],
        [ 2089,  2202,  2318,  ...,  1760,  1645,  1760]])


#### Evaluate User Diversity Preference Scale

In [8]:
user_dps_df = evaluator.eval_user_diversity_preference_scale(encoded_train_df, feature_engineer.vocab2idx, normalized=True)
user_dps_df.head(1)

Calculating user diversity preference scale:   0%|          | 0/2064 [00:00<?, ?it/s]

Calculating user diversity preference scale: 100%|██████████| 2064/2064 [00:06<00:00, 339.09it/s]


,userID,actorID_wvec,actorID_dps,country_wvec,country_dps,directorID_wvec,directorID_dps,genre_wvec,genre_dps
0,0,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 4.0, 0.0, 0.0, ...",0.577311,"[0.0, 0.0, 0.0, 0.0, 0.0, 4.0, 0.0, 0.0, 0.0, ...",0.183214,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",0.494533,"[97.0, 41.5, 8.0, 5.0, 39.0, 45.5, 0.0, 44.5, ...",0.755545


#### Prepare Item Multihot Vec on Each Dimension

In [9]:
item_vec_df = evaluator.get_item_feature_multihot_vec(encoded_train_df, feature_engineer.vocab2idx)
item_vec_df.head(1)

,movieID,actorID_vec,country_vec,directorID_vec,genre_vec
0,1588,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[1, 1, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."


#### Prepare train/valid triplet data

In [10]:
train_triplet_df = experiment_data_preprocessor.prepare_triplet_df(encoded_train_df, k_negative_samples=5)
train_triplet_with_dps_df = train_triplet_df.merge(user_dps_df, on="userID", how="left")
train_triplet_with_dps_df = train_triplet_with_dps_df.merge(item_vec_df, left_on="pos_item_id", right_on="movieID", how="left")
# train_triplet_with_dps_df.head(1)

valid_triplet_df = experiment_data_preprocessor.prepare_triplet_df(encoded_valid_df, k_negative_samples=5)
valid_triplet_with_dps_df = valid_triplet_df.merge(user_dps_df, on="userID", how="left")
valid_triplet_with_dps_df = valid_triplet_with_dps_df.merge(item_vec_df, left_on="pos_item_id", right_on="movieID", how="inner")
valid_triplet_with_dps_df.head(1)

Original data count (positive samples): 158984
Num of triplets: 158984(pos samples) * 5(negative sampled items) = 794920
Original data count (positive samples): 18261
Num of triplets: 18261(pos samples) * 5(negative sampled items) = 91305


,userID,pos_item_id,neg_item_id,actorID_idx,country_idx,directorID_idx,genre_idx,neg_actorID_idx,neg_country_idx,neg_directorID_idx,...,country_dps,directorID_wvec,directorID_dps,genre_wvec,genre_dps,movieID,actorID_vec,country_vec,directorID_vec,genre_vec
0,0,962,4662,"[328, 704, 1, 1, 1]",37,464,"[2, 3, 6, 10, 12, 0, 0, 0]","[988, 648, 2087, 1, 924]",37,75,...,0.183214,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",0.494533,"[97.0, 41.5, 8.0, 5.0, 39.0, 45.5, 0.0, 44.5, ...",0.755545,962,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[1, 1, 0, 0, 1, 0, 0, 0, 1, 0, 1, 0, 0, 0, 0, ..."


#### Prepare prediction pool for inference/testing

In [11]:
# NOTE: Prepare prediction pool to evaluate the model
prediction_pool_df = experiment_data_preprocessor.prepare_prediction_df(encoded_test_df, K=500)
prediction_pool_df.tail()

Prediction DataFrame:
User Pool: 2064
Item Pool: 6959, negative sampled to 500 items for each user
Num of interactions: 2064(users) * 500(items) = 1032000


,userID,movieID,label,actorID_idx,country_idx,directorID_idx,genre_idx
1031995,2063,447,0,"[2136, 281, 1446, 61, 1]",36,1,"[9, 0, 0, 0, 0, 0, 0, 0]"
1031996,2063,3601,0,"[1578, 466, 911, 941, 887]",37,432,"[2, 19, 0, 0, 0, 0, 0, 0]"
1031997,2063,7982,0,"[1, 1, 1, 1, 1]",37,1,"[12, 0, 0, 0, 0, 0, 0, 0]"
1031998,2063,5969,0,"[1, 1383, 554, 1, 828]",37,534,"[9, 16, 0, 0, 0, 0, 0, 0]"
1031999,2063,5025,0,"[446, 1010, 637, 809, 547]",36,70,"[7, 12, 15, 18, 0, 0, 0, 0]"


### Prepare DataLoader

In [12]:
# NOTE: ensure reproducibility of DataLoader
import torch
from common.utils import seed_worker
g = torch.Generator()
g.manual_seed(RANDOM_SEED)

# TODO: determine which Dataset to use
from torch.utils.data import DataLoader
from common.datasets import TripletDataset, UserItemPairDataset, UserPosItemSampler, get_user_triplet_mapping

BATCH_SIZE = 1024

train_dataset = TripletDataset(train_triplet_with_dps_df)
valid_dataset = TripletDataset(valid_triplet_with_dps_df)
test_dataset = UserItemPairDataset(prediction_pool_df)
print("train data count:", len(train_dataset))
print("valid data count:", len(valid_dataset))
print("test data count:", len(test_dataset))

# TODO: Custom sampler
# MIN_POS_ITEMS = 2
# user_to_pos_items, user_pos_to_indices = get_user_triplet_mapping(train_triplet_with_dps_df, MIN_POS_ITEMS)
# train_sampler = UserPosItemSampler(user_to_pos_items, user_pos_to_indices, batch_size=BATCH_SIZE, min_pos_items=MIN_POS_ITEMS, max_pos_items=20, buffer=0)
# train_loader = DataLoader(train_dataset, batch_sampler=train_sampler, num_workers=4, worker_init_fn=seed_worker, generator=g)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, worker_init_fn=seed_worker, generator=g, num_workers=4)
valid_loader = DataLoader(valid_dataset, batch_size=BATCH_SIZE)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE)


train data count: 794920
valid data count: 90805
test data count: 1032000


### Configure Model (LightningModule)

In [13]:
from lightning_models.mtdp_ngcf_v2 import MTDPRecSRM

EMB_DIM = 64
LR = 1e-3
EPOCHS = 50
NUM_LAYERS = 3
REG_WEIGHT = 1e-5

DPS_WEIGHTS = {
    "actor_dps": 0.25,
    "country_dps": 0.25,
    "director_dps": 0.25,
    "genre_dps": 0.25,
}

DPR_WEIGHTS = {
    "actor_dpr": 0.25,
    "country_dpr": 0.25,
    "director_dpr": 0.25,
    "genre_dpr": 0.25,
}

DPM_WEIGHTS = {
    "actor_pd": 0.25,
    "country_pd": 0.25,
    "director_pd": 0.25,
    "genre_pd": 0.25,
}

MT_WEIGHTS = {
    "bpr_loss": 1.0,
    "dps_loss": 0,
    "dpr_loss": 0,
    "dpm_loss": 0,
}

RESCALE_METHOD = None  # None, "log", "ema"

model = MTDPRecSRM(
    graph_data=train_graph,  # shape [2, num_edges]
    num_users=TRAIN_NUM_USERS,
    num_items=TRAIN_NUM_ITEMS,
    embedding_dim=EMB_DIM,
    num_layers=NUM_LAYERS,
    node_dropout=0.0,
    mess_dropout=0.1,
    lr=LR,
    reg_weight=REG_WEIGHT,
    dps_weights=DPS_WEIGHTS,
    dpr_weights=DPR_WEIGHTS,
    dpm_weights=DPM_WEIGHTS,
    mt_weights=MT_WEIGHTS,
    rescale_method=RESCALE_METHOD,
)


Seed set to 42


### Configure Trainer and Experiment

In [14]:
from common._mlflow import get_mlflow_logger, get_callbacks

EXPERIMENT_NAME = "rerank"
VERSION = "ngcf_v2"
RUN_NAME = "run00"
PATIENCE = 5
mlflow_logger = get_mlflow_logger(experiment_name=EXPERIMENT_NAME, run_name=RUN_NAME, tags={"version": VERSION})
trainer_callbacks = get_callbacks(
    exp_name=EXPERIMENT_NAME,
    version_name=VERSION,
    run_name=RUN_NAME,
    patience=PATIENCE,
    monitor_metric="val_loss",
    monitor_mode="min",
    hyper_param_str=f"",
)

In [15]:
from pytorch_lightning import Trainer

trainer = Trainer(
    max_epochs=EPOCHS,
    logger=mlflow_logger,
    log_every_n_steps=50,
    callbacks=trainer_callbacks,
    accelerator='auto',  # or 'auto', 'gpu'
    # devices=[0], # if gpu is available
)


Trainer will use only 1 of 2 GPUs because it is running inside an interactive / notebook environment. You may try to set `Trainer(devices=2)` but please note that multi-GPU inside interactive / notebook environments is considered experimental and unstable. Your mileage may vary.
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs


### Train Model

In [16]:
# Start training
trainer.fit(model, train_dataloaders=train_loader, val_dataloaders=valid_loader)


/home/adam/R11_Bai/DPRecSys/.venv/lib/python3.11/site-packages/pytorch_lightning/callbacks/model_checkpoint.py:654: Checkpoint directory /home/adam/R11_Bai/DPRecSys/experiments/test_checkpoints/rerank exists and is not empty.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]

   | Name                | Type              | Params | Mode 
-------------------------------------------------------------------
0  | ngcf_model          | NGCF              | 714 K  | train
1  | bpr_loss            | BPRLoss           | 0      | train
2  | reg_loss            | EmbLoss           | 0      | train
3  | dps_module          | DPSPredictor      | 1.0 K  | train
4  | dps_loss_fn         | DPSLoss           | 0      | train
5  | dpr_module          | DPRegularizer     | 263 K  | train
6  | dpr_loss_fn         | DPRLoss           | 0      | train
7  | dpm_module          | DPMatcher         | 0      | train
8  | dpm_loss_fn         | KLDivergenceLoss  | 0      | train
9  | loss_scaling_module | LogScaleLoss  

Sanity Checking: |          | 0/? [00:00<?, ?it/s]

/home/adam/R11_Bai/DPRecSys/.venv/lib/python3.11/site-packages/pytorch_lightning/trainer/connectors/data_connector.py:425: The 'val_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=15` in the `DataLoader` to improve performance.


Training: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

...

Validation: |          | 0/? [00:00<?, ?it/s]

Epoch 4, global step 3885: 'val_loss' was not in top 1


Validation: |          | 0/? [00:00<?, ?it/s]

Monitored metric val_loss did not improve in the last 5 records. Best score: 0.595. Signaling Trainer to stop.
Epoch 5, global step 4662: 'val_loss' was not in top 1


🏃 View run run00 at: http://140.112.106.216:3683/#/experiments/19/runs/f908b3af600f4e90a31220800aba9654
🧪 View experiment at: http://140.112.106.216:3683/#/experiments/19


### Inference

In [17]:
# NOTE: the inference model MUST be the same as the training model
# best_model_experiment_name = "rerank"
# best_model_checkpoint_path = "[ngcf_v2]-test_run--best-checkpoint-epoch=00-val_loss=2.93.ckpt"
# best_model_path = f"test_checkpoints/{best_model_experiment_name}/{best_model_checkpoint_path}"
best_model_path = trainer.checkpoint_callback.best_model_path

model = MTDPRecSRM.load_from_checkpoint(checkpoint_path=best_model_path)
# start inference
trainer.test(model=model, dataloaders=test_loader)


Seed set to 42
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]
/home/adam/R11_Bai/DPRecSys/.venv/lib/python3.11/site-packages/pytorch_lightning/trainer/connectors/data_connector.py:425: The 'test_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=15` in the `DataLoader` to improve performance.


Testing: |          | 0/? [00:00<?, ?it/s]

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃        Test metric        ┃       DataLoader 0        ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│        test_ndcg10        │    0.3693173825740814     │
│        test_ndcg20        │    0.39844560623168945    │
│        test_ndcg5         │    0.32048583030700684    │
│     test_precision10      │    0.1355135589838028     │
│     test_precision20      │    0.11654554307460785    │
│      test_precision5      │    0.1472868174314499     │
│       test_recall10       │    0.11921688169240952    │
│       test_recall20       │    0.19435930252075195    │
│       test_recall5        │    0.06696981936693192    │
└───────────────────────────┴───────────────────────────┘

🏃 View run run00 at: http://140.112.106.216:3683/#/experiments/19/runs/f908b3af600f4e90a31220800aba9654
🧪 View experiment at: http://140.112.106.216:3683/#/experiments/19


[{'test_ndcg5': 0.32048583030700684,
  'test_ndcg10': 0.3693173825740814,
  'test_ndcg20': 0.39844560623168945,
  'test_precision5': 0.1472868174314499,
  'test_precision10': 0.1355135589838028,
  'test_precision20': 0.11654554307460785,
  'test_recall5': 0.06696981936693192,
  'test_recall10': 0.11921688169240952,
  'test_recall20': 0.19435930252075195}]

In [18]:
model.test_results["eval_score_df"].describe()

,user,ndcg@5,recall@5,precision@5,ndcg@10,recall@10,precision@10,ndcg@20,recall@20,precision@20
count,2064.000000,2064.000000,2064.000000,2064.000000,2064.000000,2064.000000,2064.000000,2064.000000,2064.000000,2064.000000
mean,1031.500000,0.320486,0.066970,0.147287,0.369317,0.119217,0.135514,0.398446,0.194359,0.116546
std,595.969798,0.365965,0.116284,0.189426,0.323787,0.155684,0.143812,0.280284,0.193950,0.110165
min,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,515.750000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.235409,0.041667,0.050000
50%,1031.500000,0.000000,0.000000,0.000000,0.386853,0.073734,0.100000,0.401733,0.153010,0.100000
75%,1547.250000,0.630930,0.090909,0.200000,0.618171,0.176471,0.200000,0.594447,0.285714,0.200000
max,2063.000000,1.000000,1.000000,1.000000,1.000000,1.000000,0.900000,1.000000,1.000000,0.800000


In [19]:
prefix = "(mt)ngcf_v2_k5"
embedding_path = "embeddings/"

torch.save(model.user_emb.cpu(), f"{embedding_path}{prefix}_user_emb.pt")
torch.save(model.item_emb.cpu(), f"{embedding_path}{prefix}_item_emb.pt")

In [20]:
# Evaluate user diversity preference matching score (DPMS) at k
eval_df = evaluator.prepare_evaluation_data(model.test_results, feature_engineer.idx2vocab)

# Get user DPMS
user_dpms_df = evaluator.evaluate_dpms_at_k(
    eval_df=eval_df,
    feature_engineer=feature_engineer,
    ground_truth_dps_df=user_dps_df,
    k=20,
    actor_k=5,
    rare_threshold=5,
)

user_dpms_df.describe()

exploded 2064
extracting item features...
merging features...
interaction data count before merging: 41280
interaction data count after merging: 41280
done!
Transformed: Re-index user/item mapping
Transformed: Encoded idx for actorID
Transformed: Encoded idx for country
Transformed: Encoded idx for directorID
Transformed: Encoded idx for genre
encoded 2064


Calculating user diversity preference scale: 100%|██████████| 2064/2064 [00:03<00:00, 685.68it/s]


combined_df 2064
Calculating DPMS for each feature...


,userID,actorID_dpms,country_dpms,directorID_dpms,genre_dpms,avg_dpms
count,2064.000000,2064.000000,2064.000000,2064.000000,2064.000000,2064.000000
mean,1031.500000,0.163257,0.975495,0.160620,0.884718,0.546023
std,595.969798,0.074057,0.033267,0.101668,0.074649,0.052788
min,0.000000,0.000000,0.438207,0.000000,0.386221,0.298375
25%,515.750000,0.107092,0.972216,0.082536,0.856962,0.511858
50%,1031.500000,0.163787,0.984858,0.155008,0.905553,0.550741
75%,1547.250000,0.218617,0.991439,0.230501,0.934801,0.583744
max,2063.000000,0.375640,0.999578,0.569165,0.984803,0.701302


In [21]:
import joblib
dir = "tmp"
if not os.path.exists(dir):
    os.makedirs(dir)
joblib.dump(eval_df, os.path.join(dir, "eval_df.pkl"))
joblib.dump(user_dps_df, os.path.join(dir, "user_dps_df.pkl"))
joblib.dump(feature_engineer, os.path.join(dir, "feature_engineer.pkl"))

['tmp/feature_engineer.pkl']

In [1]:
import os
import sys

sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))

import joblib
dir = "tmp"
eval_df = joblib.load(os.path.join(dir, "eval_df.pkl"))
user_dps_df = joblib.load(os.path.join(dir, "user_dps_df.pkl"))
feature_engineer = joblib.load(os.path.join(dir, "feature_engineer.pkl"))

/home/adam/R11_Bai/DPRecSys/.venv/lib/python3.11/site-packages/transformers/utils/generic.py:441: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(


In [2]:
from post_processing.dpa_rs import DPA_RS

reranker = DPA_RS(
    eval_df=eval_df,
    feature_engineer=feature_engineer,
    m=100,
    ground_truth_dps_df=user_dps_df
)


Seed set to 42


In [ ]:
result_df = reranker.rerank(
    top_k=20,
    max_iter=100,
    random_state=42,
)
result_df.head()

Preparing input DataFrame for DPA-RS...
exploded 2064
extracting item features...
merging features...
interaction data count before merging: 206400
interaction data count after merging: 206400
done!
Transformed: Re-index user/item mapping
Transformed: Encoded idx for actorID
Transformed: Encoded idx for country
Transformed: Encoded idx for directorID
Transformed: Encoded idx for genre
encoded 2064
Combined DataFrame shape: (206400, 15)
Reranking items for each user...


Reranking users:   0%|          | 0/2064 [00:00<?, ?it/s](CVXPY) Jun 21 07:40:13 PM: Your problem has 100 variables, 201 constraints, and 0 parameters.
(CVXPY) Jun 21 07:40:13 PM: It is compliant with the following grammars: DCP, DQCP
(CVXPY) Jun 21 07:40:13 PM: (If you need to solve this problem multiple times, but with different data, consider using parameters.)
(CVXPY) Jun 21 07:40:13 PM: CVXPY will first compile your problem; then, it will invoke a numerical solver to obtain a solution.
(CVXPY) Jun 21 07:40:13 PM: Your problem is compiled with the CPP canonicalization backend.
(CVXPY) Jun 21 07:40:13 PM: Compiling problem (target solver=ECOS).
(CVXPY) Jun 21 07:40:13 PM: Reduction chain: FlipObjective -> Dcp2Cone -> CvxAttr2Constr -> ConeMatrixStuffing -> ECOS
(CVXPY) Jun 21 07:40:13 PM: Applying reduction FlipObjective
(CVXPY) Jun 21 07:40:13 PM: Applying reduction Dcp2Cone
(CVXPY) Jun 21 07:40:13 PM: Applying reduction CvxAttr2Constr
(CVXPY) Jun 21 07:40:13 PM: Applying reduction

                                     CVXPY                                     
                                     v1.6.6                                    
-------------------------------------------------------------------------------
                                  Compilation                                  
-------------------------------------------------------------------------------
-------------------------------------------------------------------------------
                                Numerical solver                               
-------------------------------------------------------------------------------

ECOS 2.0.10 - (C) embotech GmbH, Zurich Switzerland, 2012-15. Web: www.embotech.com/ECOS

It     pcost       dcost      gap   pres   dres    k/t    mu     step   sigma     IR    |   BT
 0  -1.205e+01  -1.585e+02  +9e+02  6e-01  2e-02  1e+00  4e+00    ---    ---    1  1  - |  -  - 
 1  +2.232e+00  -7.943e+01  +8e+02  3e-01  1e-02  1e+00  4e+00  0.2509  4e-01

(CVXPY) Jun 21 07:40:13 PM: Problem status: optimal
(CVXPY) Jun 21 07:40:13 PM: Optimal value: 7.371e-08
(CVXPY) Jun 21 07:40:13 PM: Compilation took 1.066e-02 seconds
(CVXPY) Jun 21 07:40:13 PM: Solver (including time spent in interface) took 9.909e-03 seconds
(CVXPY) Jun 21 07:40:13 PM: Your problem has 100 variables, 201 constraints, and 0 parameters.
(CVXPY) Jun 21 07:40:13 PM: It is compliant with the following grammars: DCP, DQCP
(CVXPY) Jun 21 07:40:13 PM: (If you need to solve this problem multiple times, but with different data, consider using parameters.)
(CVXPY) Jun 21 07:40:13 PM: CVXPY will first compile your problem; then, it will invoke a numerical solver to obtain a solution.
(CVXPY) Jun 21 07:40:13 PM: Your problem is compiled with the CPP canonicalization backend.
(CVXPY) Jun 21 07:40:13 PM: Compiling problem (target solver=ECOS).
(CVXPY) Jun 21 07:40:13 PM: Reduction chain: FlipObjective -> Dcp2Cone -> CvxAttr2Constr -> ConeMatrixStuffing -> ECOS
(CVXPY) Jun 21 07:40


ECOS 2.0.10 - (C) embotech GmbH, Zurich Switzerland, 2012-15. Web: www.embotech.com/ECOS

It     pcost       dcost      gap   pres   dres    k/t    mu     step   sigma     IR    |   BT
 0  -1.687e+00  -1.087e+02  +6e+02  6e-01  3e-02  1e+00  3e+00    ---    ---    1  1  - |  -  - 
 1  +4.713e+00  -3.202e+01  +5e+02  2e-01  1e-02  1e+00  2e+00  0.3950  3e-01   1  1  1 |  0  0
 2  +7.440e-01  -2.823e+00  +9e+01  2e-02  8e-04  3e-01  4e-01  0.8793  9e-02   1  1  1 |  0  0
 3  +2.164e-01  -1.736e+00  +5e+01  9e-03  4e-04  1e-01  3e-01  0.5355  2e-01   2  2  2 |  0  0
 4  +5.357e-02  -6.184e-01  +2e+01  3e-03  1e-04  4e-02  1e-01  0.6783  7e-02   1  2  2 |  0  0
 5  +1.269e-03  -5.121e-01  +1e+01  2e-03  8e-05  2e-02  7e-02  0.5640  6e-01   1  2  1 |  0  0
 6  -6.152e-03  -2.728e-01  +7e+00  1e-03  4e-05  4e-03  4e-02  0.9019  5e-01   1  2  2 |  0  0
 7  -1.967e-03  -8.527e-02  +2e+00  3e-04  1e-05  8e-04  1e-02  0.8848  2e-01   1  2  1 |  0  0
 8  +4.484e-04  -2.569e-02  +7e-01  9e-05  3e

...

(CVXPY) Jun 21 07:40:16 PM: It is compliant with the following grammars: DCP, DQCP
(CVXPY) Jun 21 07:40:16 PM: (If you need to solve this problem multiple times, but with different data, consider using parameters.)
(CVXPY) Jun 21 07:40:16 PM: CVXPY will first compile your problem; then, it will invoke a numerical solver to obtain a solution.
(CVXPY) Jun 21 07:40:16 PM: Your problem is compiled with the CPP canonicalization backend.
(CVXPY) Jun 21 07:40:16 PM: Compiling problem (target solver=ECOS).
(CVXPY) Jun 21 07:40:16 PM: Reduction chain: FlipObjective -> Dcp2Cone -> CvxAttr2Constr -> ConeMatrixStuffing -> ECOS
(CVXPY) Jun 21 07:40:16 PM: Applying reduction FlipObjective
(CVXPY) Jun 21 07:40:16 PM: Applying reduction Dcp2Cone
(CVXPY) Jun 21 07:40:16 PM: Applying reduction CvxAttr2Constr
(CVXPY) Jun 21 07:40:16 PM: Applying reduction ConeMatrixStuffing
(CVXPY) Jun 21 07:40:16 PM: Applying reduction ECOS
(CVXPY) Jun 21 07:40:16 PM: Finished problem compilation (took 1.044e-02 seconds

-------------------------------------------------------------------------------
                                  Compilation                                  
-------------------------------------------------------------------------------
-------------------------------------------------------------------------------
                                Numerical solver                               
-------------------------------------------------------------------------------

ECOS 2.0.10 - (C) embotech GmbH, Zurich Switzerland, 2012-15. Web: www.embotech.com/ECOS

It     pcost       dcost      gap   pres   dres    k/t    mu     step   sigma     IR    |   BT
 0  -2.510e+00  -1.160e+02  +7e+02  6e-01  2e-02  1e+00  3e+00    ---    ---    1  1  - |  -  - 
 1  +3.449e-01  -2.811e+01  +4e+02  2e-01  4e-03  5e-01  2e+00  0.5569  2e-01   2  2  1 |  0  0
 2  -4.070e-01  -6.004e+00  +1e+02  3e-02  5e-04  1e-01  5e-01  0.7437  6e-02   2  2  2 |  0  0
 3  -1.508e-01  -1.181e+00  +2e+01  5e-03  7e

(CVXPY) Jun 21 07:40:16 PM: It is compliant with the following grammars: DCP, DQCP
(CVXPY) Jun 21 07:40:16 PM: (If you need to solve this problem multiple times, but with different data, consider using parameters.)
(CVXPY) Jun 21 07:40:16 PM: CVXPY will first compile your problem; then, it will invoke a numerical solver to obtain a solution.
(CVXPY) Jun 21 07:40:16 PM: Your problem is compiled with the CPP canonicalization backend.
(CVXPY) Jun 21 07:40:16 PM: Compiling problem (target solver=ECOS).
(CVXPY) Jun 21 07:40:16 PM: Reduction chain: FlipObjective -> Dcp2Cone -> CvxAttr2Constr -> ConeMatrixStuffing -> ECOS
(CVXPY) Jun 21 07:40:16 PM: Applying reduction FlipObjective
(CVXPY) Jun 21 07:40:16 PM: Applying reduction Dcp2Cone
(CVXPY) Jun 21 07:40:16 PM: Applying reduction CvxAttr2Constr
(CVXPY) Jun 21 07:40:16 PM: Applying reduction ConeMatrixStuffing
(CVXPY) Jun 21 07:40:16 PM: Applying reduction ECOS
(CVXPY) Jun 21 07:40:16 PM: Finished problem compilation (took 9.786e-03 seconds

-------------------------------------------------------------------------------
                                  Compilation                                  
-------------------------------------------------------------------------------
-------------------------------------------------------------------------------
                                Numerical solver                               
-------------------------------------------------------------------------------

ECOS 2.0.10 - (C) embotech GmbH, Zurich Switzerland, 2012-15. Web: www.embotech.com/ECOS

It     pcost       dcost      gap   pres   dres    k/t    mu     step   sigma     IR    |   BT
 0  -1.855e+00  -1.119e+02  +6e+02  6e-01  3e-02  1e+00  3e+00    ---    ---    1  1  - |  -  - 
 1  +4.259e+00  -2.898e+01  +4e+02  2e-01  7e-03  9e-01  2e+00  0.4708  2e-01   1  1  1 |  0  0
 2  +2.320e-01  -3.018e+00  +6e+01  2e-02  5e-04  1e-01  3e-01  0.8802  6e-02   1  1  2 |  0  0
 3  +2.704e-02  -9.225e-01  +2e+01  4e-03  1e

(CVXPY) Jun 21 07:40:16 PM: Finished problem compilation (took 1.028e-02 seconds).
(CVXPY) Jun 21 07:40:16 PM: Invoking solver ECOS  to obtain a solution.
(CVXPY) Jun 21 07:40:16 PM: Problem status: optimal
(CVXPY) Jun 21 07:40:16 PM: Optimal value: 5.978e-03
(CVXPY) Jun 21 07:40:16 PM: Compilation took 1.028e-02 seconds
(CVXPY) Jun 21 07:40:16 PM: Solver (including time spent in interface) took 8.043e-03 seconds
(CVXPY) Jun 21 07:40:16 PM: Your problem has 100 variables, 201 constraints, and 0 parameters.
(CVXPY) Jun 21 07:40:16 PM: It is compliant with the following grammars: DCP, DQCP
(CVXPY) Jun 21 07:40:16 PM: (If you need to solve this problem multiple times, but with different data, consider using parameters.)
(CVXPY) Jun 21 07:40:16 PM: CVXPY will first compile your problem; then, it will invoke a numerical solver to obtain a solution.
(CVXPY) Jun 21 07:40:16 PM: Your problem is compiled with the CPP canonicalization backend.
(CVXPY) Jun 21 07:40:16 PM: Compiling problem (targe

In [ ]:
reranked_score_df = evaluator.evaluate(result_df, K=5)
reranked_score_df = evaluator.evaluate(reranked_score_df, K=10)
reranked_score_df = evaluator.evaluate(reranked_score_df, K=20)
reranked_score_df.describe()

,user,ndcg@5,recall@5,precision@5,ndcg@10,recall@10,precision@10,ndcg@20,recall@20,precision@20
count,2064.000000,2064.000000,2064.000000,2064.000000,2064.000000,2064.000000,2064.000000,2064.000000,2064.000000,2064.000000
mean,35564.367733,0.228575,0.042411,0.094477,0.285809,0.078247,0.092442,0.335594,0.151584,0.092902
std,20797.975208,0.331606,0.087950,0.142984,0.305457,0.119547,0.110308,0.257568,0.162616,0.091146
min,75.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,17798.500000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
50%,35054.000000,0.000000,0.000000,0.000000,0.301030,0.036376,0.100000,0.336699,0.115832,0.050000
75%,53331.000000,0.500000,0.052632,0.200000,0.500000,0.111111,0.100000,0.500000,0.222222,0.150000
max,71534.000000,1.000000,1.000000,0.800000,1.000000,1.000000,0.600000,1.000000,1.000000,0.500000


In [ ]:
# from common.eval import Evaluator
# evaluator = Evaluator()
reranked_user_dpms_df = evaluator.evaluate_dpms_at_k(
    eval_df=result_df,
    feature_engineer=feature_engineer,
    ground_truth_dps_df=user_dps_df,
    k=20,
    actor_k=5,
    rare_threshold=5,
)

reranked_user_dpms_df.describe()

exploded 2064
extracting item features...
merging features...
interaction data count before merging: 41280
interaction data count after merging: 41280
done!
Transformed: Re-index user/item mapping
Transformed: Encoded idx for actorID
Transformed: Encoded idx for country
Transformed: Encoded idx for directorID
Transformed: Encoded idx for genre
encoded 2064


Calculating user diversity preference scale: 100%|██████████| 2064/2064 [00:03<00:00, 668.73it/s]


combined_df 2064
Calculating DPMS for each feature...


,userID,actorID_dpms,country_dpms,directorID_dpms,genre_dpms,avg_dpms
count,2064.000000,2064.000000,2064.000000,2064.000000,2064.000000,2064.000000
mean,1031.500000,0.300113,0.991161,0.394626,0.959495,0.661349
std,595.969798,0.072029,0.010848,0.134547,0.019566,0.046398
min,0.000000,0.030438,0.799220,0.000000,0.762575,0.477504
25%,515.750000,0.253988,0.989440,0.320172,0.950511,0.633871
50%,1031.500000,0.304789,0.993768,0.410647,0.962659,0.666983
75%,1547.250000,0.349773,0.996482,0.486287,0.973283,0.694167
max,2063.000000,0.543348,0.999974,0.752244,0.992558,0.785857
